<a href="https://colab.research.google.com/github/jolineuichanco/DataAnalytics/blob/main/demos/Optimization_Workshop_(Class).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Content adapted from Holly Wiberg's notebook

Begin by copying this notebook. Go to File and choose Save a copy in Drive (If you don't see the file tab at the top, use the arrow at the top right of your window to make the header visible). If you're not logged into a google account, it will ask you to do so. Then a new tab should open with a new file called "Copy of Optimization_Workshop_(Class)". That is the file in which you will work for your pre-assignment.
To begin, we need to install guroby and load our license!

In [ ]:
!pip install gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import Env, Model, GRB

env = Env(empty=True)
env.setParam("WLSACCESSID", "xxxxxx-xxxx-xxxx-xxxx-xxxxxxxxx")  # Replace with your Access ID
env.setParam("WLSSECRET", "xxxxxx-xxxx-xxxx-xxxx-xxxxxxxx")  # Replace with your Secret Key
env.setParam("LICENSEID", 1234567)  # Replace the number with your license ID
env.start()  # Start the environment

##A First Example

Let's see how we translate a simple, 2 variable LP to python code.

\begin{align*}
\max_{x, y} &\: x + 2y\\
 s.t. &  \: \:x + y \leq 1\\
& x, y \geq 0 .
\end{align*}

We first need to construct a model object. This is a container for everything in our optimization problem: variables, constraints, solver options, etc. If you're experimenting with an optimization model, it's generally a good idea to set a short time limit for the solver calculations. This will (hopefully) force the solver to stop running if you accidentally pass it a very large model that it can't solve.

In [ ]:
model = gp.Model()
model.setParam("TimeLimit", 60)

Next, we define the two decision variables in our optimization problem. We will use the .addVar function. The "lb" argument specifies the variable lower bounds, and the second one specifies the name of the variable. Note that we could also set upper bounds with the argument "ub".

In [ ]:
x = model.addVar(lb=0, name="x")
y = model.addVar(lb=0, name="y")

You can check the summary of the model by calling the .update() function. In these notebooks, when you run a cell it prints the output of the last line. For instance, we can print the model object by adding a line at the bottom that simply calls the model.

In [ ]:
model.update()
model

We now add the single constraint of our problem using the .addConstr function. We write it algebraically, exactly as we see it above and we name it "c1"

In [ ]:
model.addConstr(x + y <= 1, "c1")
model.update()
model

We next specify the objective with the setObjective function. It is important to indicate if we are solving a minimization or a maximization problem!

In [ ]:
model.setObjective(x + 2 * y, gp.GRB.MAXIMIZE)

In [ ]:
model.update()
model

Lastly, to solve the optimization problem, call the .optimize() function.

In [ ]:
# Optimize the model
model.optimize()

# Get termination status
status = model.Status

# Print status
print("Optimization Status:", status)

The meaning of the termination status can be found [here](https://docs.gurobi.com/projects/optimizer/en/current/reference/numericcodes/statuscodes.html)

For instance:

2. (gp.GRB.OPTIMAL): The optimal solution was found.

3. (gp.GRB.INFEASIBLE): The model is infeasible (no solution satisfies all constraints).

4. (gp.GRB.INF_OR_UNBD): The model is infeasible or unbounded.

5. (gp.GRB.UNBOUNDED): The model is unbounded (objective can increase indefinitely).

9. (gp.GRB.TIME_LIMIT): Optimization stopped because the time limit was reached.

11. (gp.GRB.INTERRUPTED): Optimization was interrupted by the user.

We can obtain the optimal solution found for each variable by calling the .X attribute. For the optimal objetcive value, we call the .ObjVal attribute as below:

In [ ]:
x_value = x.X
x_value

In [ ]:
y_value = y.X
y_value

In [ ]:
objective_value = model.ObjVal
objective_value

## Exercise 1

Code and solve the following optimization problem:

\begin{aligned}
    \min_{x, y} \quad & 3x - y \\
    \text{s.t.} \quad & x + 2y \geq 1, \\
    & x \geq 0, \\
    & 0 \leq y \leq 1.
\end{aligned}

In [ ]:
#WRITE YOUR CODE HERE



Take the problem from exercise 1 and make two copies below.

In the first copy, add a new constraint to make the problem **infeasible** (i.e., there are no values of x and y that satisfy all the constraints)
In the second copy, change the constraints or the objective function to make the problem **unbounded** (i.e., the optimal solution is infinite)

Solve both versions of the problem and look at the termination_status to see if you have succeeded.

In [ ]:
#INFEASIBLE PROBLEM HERE


In [ ]:
#UNBOUNDED PROBLEM HERE


## Constructing the Olympics "Dream Team"
Now, we're ready to start using optimization to formulate and solve real problems. Suppose that we want to design the optimal Basketball team for the Olympics.

First, let's load our 2018-2019 player data. This CSV file contains statistics for active US-born players in the 2018-2019 season. We have also added in information about whether each player appeared in the previous Olympics (2016) and the most recent NBA All-Star game as additional measures.

Source: https://www.basketball-reference.com/

In [3]:
import pandas as pd

# Read CSV file into a DataFrame
url = 'https://raw.githubusercontent.com/hwiberg/SSAC2020/master/NBA_data_2018_2019.csv'
df = pd.read_csv(url)

Let's see what this data looks like:

In [ ]:
#Load the first 10 rows
df.head(10)

## A Simple Model
We'll start by coding up a simple model.

In [ ]:
# Initialize the model with a time limit of 60 seconds
model = gp.Model()
model.setParam("TimeLimit", 60)

### I. Defining the Decision Variables
First we need to create our decision variables. Remember, we want to construct variables:

$$
x_i =
\begin{cases}
    1 & \text{if player } i \text{ is selected for the team,} \\
    0 & \text{otherwise.}
\end{cases}
$$

These are called binary variables since they only take on values between 0 and 1. We need an
$x_i$ for all $i=1,..., N$.

In [ ]:
# Get number of rows in the DataFrame
N = len(df)

# Add binary variables x[1] to x[N]
x = model.addVars(N, vtype=gp.GRB.BINARY, name="x")

### II. Formalizing the Objective
We'll start with a basic objective, which is simply to maximize the average Player Efficiency Rating (PER) of the players on the selected team. We'll denote player $i$'s PER as $s_i$


We can then calculate the objective as:

$$
\frac{1}{12} \sum_{i=1}^{N} (x_i * s_i)
$$




We can formulate this as follows:

In [ ]:
# Extract the 'PER' column as a list or series
s = df["PER"].tolist()  # Convert to list for indexing
s

In [ ]:
# Set objective function
model.setObjective((1/12) * sum(x[i] * s[i] for i in range(N)), gp.GRB.MAXIMIZE)

### III. Constructing the Constraints
Now that we have defined our variables and quantified our goal, we need to specify what requirements any team must satisfy. Let's start by just placing a constraint on team size.

In [ ]:
# Add constraint: sum of selected players must be exactly 12
model.addConstr(sum(x[i] for i in range(N)) == 12, "team_size")

### IV. Solving the Model
We have specified the three key elements of any mixed-integer optimization model:

Decision Variables

*   Decision Variables
*   Objective
*   Feasibility Constraints


We're ready to solve the model!

In [ ]:
model.update()
model

In [ ]:
# Solve the optimization model
model.optimize()

Let's see what our results look like: we can look at the values of our decision variables $x_1,..., x_N$
 to see which indices were set to 1. These indices correspond to the players that were selected for the team.

In [ ]:
# Retrieve the values of x after optimization
selection_indices = [x[i].X for i in range(N)]
selection_indices

In [ ]:
# Filter the dataframe to get the selected players
selected_players = df.loc[[i for i in range(N) if selection_indices[i] == 1], "Player"]
selected_players

In [ ]:
model.ObjVal

How well does this match the list of finalists for the Olympics?

In [ ]:
# Filter the dataframe to get the selected players and their attributes
selected_players = df.loc[[i for i in range(N) if selection_indices[i] == 1],
                          ["Player", "Pos", "AllStar", "Olympics2016", "OlympicsFinalist"]]
selected_players

## Making a More Realistic Team
While this team is technically legal under Olympic regulations, it's not enough for us just to look at PER! We want to make sure that we have enough players of each position to fill out the team. We can add a few constraints to make a more useful team.

In [ ]:
# Initialize the original model
model2 = gp.Model()

# Add binary decision variables x[1:N]
x = model2.addVars(N, vtype=gp.GRB.BINARY, name="x")

# Set objective function: maximize (1/12) * sum(x[i] * df["PER"][i] for i in range(N))
model2.setObjective((1/12) * sum(x[i] * df.loc[i, "PER"] for i in range(N)), gp.GRB.MAXIMIZE)

# Add constraint: exactly 12 players must be selected
model2.addConstr(sum(x[i] for i in range(N)) == 12, "team_size")

### Position Requirements

Suppose we want at least 4 guards, 4 forwards, and 3 centers. We first define indicators for each player's position. For example, to indicate which players are forwards, we define:

$$
f_i =
\begin{cases}
    1 & \text{if player } i \text{ is a forward,} \\
    0 & \text{otherwise.}
\end{cases}
$$


We can create similar variables
 and
 for center and guard positions. Note that this is data (**parameters**); these are not decision variables.

In [ ]:
# Create binary indicators for positions
c = (df["Pos"] == "C").astype(int)       # 1 if position is "C" (Center), else 0
f = ((df["Pos"] == "PF") | (df["Pos"] == "SF")).astype(int)  # 1 if position is "PF" or "SF" (Forward), else 0
g = ((df["Pos"] == "PG") | (df["Pos"] == "SG")).astype(int)  # 1 if position is "PG" or "SG" (Guard), else 0

We can create constraints by summing the position indicator over the players who are selected for the team, like we did with the objective:
$$
\sum_{i = 1}^N(f_i * x_i)\geq 4
$$


We can do the same for the center and guard positions as well.

In [ ]:
# Add forward constraint: at least 4 forwards must be selected
model2.addConstr(sum(f[i] * x[i] for i in range(N)) >= 4, "forward_constraint")

# Add center constraint: at least 3 centers must be selected
model2.addConstr(sum(c[i] * x[i] for i in range(N)) >= 3, "center_constraint")

# Add guard constraint: at least 4 guards must be selected
model2.addConstr(sum(g[i] * x[i] for i in range(N)) >= 4, "guard_constraint")

### Defensive Ability
We want to make sure that our team has good average defensive ability. One (imperfect!) way to measure this is using Defensive Box Plus/Minus (DBPM). We can require that the average DBPM of the chosen team is at least +1.

$$
\frac{1}{12}\sum_{i = 1}^N (d_i * x_i) \geq 1
$$


where
 is player
's DBPM.

In [ ]:
# Extract the 'DBPM' column as a list
d = df["DBPM"].tolist()

# Add constraint: average DBPM of selected players must be at least 1.0
model2.addConstr((1/12) * sum(d[i] * x[i] for i in range(N)) >= 1.0, "dbpm_constraint")

### Experience Constraints

We also want to make sure our team has enough Olympics experience: let's add a constraint that at least 3 selected players were on the 2016 Olympic team.

We use a similar syntax as before:

 $$
\sum_{i=1}^N (o_i * x_i) \geq 3
 $$

where
 if player
 was on the 2016 Olympic team, and 0 otherwise.

In [ ]:
# Extract the 'Olympics2016' column as a list
o = df["Olympics2016"].tolist()

# Add constraint: at least 3 selected players must have participated in the 2016 Olympics
model2.addConstr(sum(o[i] * x[i] for i in range(N)) >= 3, "olympics2016_constraint")

Now let's solve our new model!

In [ ]:
# Set Gurobi to run in silent mode (suppress output)
model2.setParam("OutputFlag", 0)

# Optimize the model
model2.optimize()

In [ ]:
# Retrieve selected players
selected_players2 = df.loc[[i for i in range(N) if x[i].X == 1], "Player"].sort_values().tolist()
selected_players2

If we compare to the original player list, we see that we have removed Montrezl Harrell (Center) and added Jimmy Butler (Forward).

In [ ]:
selected_players = selected_players.sort_values('Player')
selected_players

How good is our new model, according to our objective function?

In [ ]:
model2.ObjVal

In [ ]:
model.ObjVal

This is < 0.1 worse than our original solution: even with these new constraints, we haven't lost much.

## Variations of Our Basic Model

Before we experiment further, we can put our full model into a function that will let us easily modify the model for different objectives and datasets.

In [ ]:
def create_roster(df, objective_metric):
    # Initialize model
    m = gp.Model()
    m.setParam("OutputFlag", 0)  # Set silent mode

    # Define number of players
    N = len(df)

    # Define binary decision variables
    x = m.addVars(N, vtype=gp.GRB.BINARY, name="x")

    # Define relevant data columns
    c = (df["Pos"] == "C").astype(int).tolist()
    f = ((df["Pos"] == "PF") | (df["Pos"] == "SF")).astype(int).tolist()
    g = ((df["Pos"] == "PG") | (df["Pos"] == "SG")).astype(int).tolist()
    d = df["DBPM"].tolist()
    o = df["Olympics2016"].tolist()

    # Pull in data for our chosen objective
    s = df[objective_metric].tolist()

    # Objective function
    m.setObjective((1/12) * sum(x[i] * s[i] for i in range(N)), gp.GRB.MAXIMIZE)

    # Team size constraint
    m.addConstr(sum(x[i] for i in range(N)) == 12, "team_size")

    # Position constraints
    m.addConstr(sum(f[i] * x[i] for i in range(N)) >= 4, "forward_constraint")
    m.addConstr(sum(c[i] * x[i] for i in range(N)) >= 3, "center_constraint")
    m.addConstr(sum(g[i] * x[i] for i in range(N)) >= 4, "guard_constraint")

    # Experience constraint
    m.addConstr(sum(o[i] * x[i] for i in range(N)) >= 3, "olympics_constraint")

    # Defensive ability constraint
    m.addConstr((1/12) * sum(d[i] * x[i] for i in range(N)) >= 1, "defense_constraint")

    # Optimize model
    m.optimize()

    # Retrieve selected players
    selected_players = df.loc[[i for i in range(N) if x[i].X > 0.99], "Player"].sort_values().tolist()

    return m.ObjVal, selected_players

Now, it is easy for us to build models for different objective functions. We simply have to specify our dataset of interest and the column of the metric that we want to maximize.

### Alternative Objective Functions
We chose to optimize average PER, but we could have chosen other metrics too. How does our team composition change when we use other measures of player performance?

In [ ]:
obj_per, players_per = create_roster(df, "PER")
obj_per, players_per

In [ ]:
obj_bpm, players_bpm = create_roster(df, "BPM")
obj_bpm, players_bpm

In [ ]:
obj_ws48, players_ws48 = create_roster(df, "WS_48")
obj_ws48, players_ws48

###. How different is the playoff perspective?

Now let's repeat the model (with the original PER objective), but using data from the past two playoff seasons instead. We're looking at average playoff statistics for all active US-born NBA players.

In [ ]:
# Read the CSV file into a DataFrame
url = 'https://raw.githubusercontent.com/hwiberg/SSAC2020/master/NBA_data_playoffs_2017_2019.csv'
df_playoffs = pd.read_csv(url)
df_playoffs.head(10)

We can leverage the same function, but now we want to speficy a new dataset, **df_playoffs**, rather than the original dataset **df**.

In [ ]:
# Run the create_roster function using "PER" as the objective metric
obj_playoff, players_playoff = create_roster(df_playoffs, "PER")

We can create a table to compare how much these different lists overlap. We'll do this using the join function.

In [ ]:
# Create individual DataFrames with player selections
df_per = pd.DataFrame({"Player": players_per, "PER": 1})
df_bpm = pd.DataFrame({"Player": players_bpm, "BPM": 1})
df_ws48 = pd.DataFrame({"Player": players_ws48, "WS_48": 1})
df_playoff = pd.DataFrame({"Player": players_playoff, "PER_PLAYOFF": 1})

# Perform an outer join on "Player"
player_compare = df_per.merge(df_bpm, on="Player", how="outer") \
                       .merge(df_ws48, on="Player", how="outer") \
                       .merge(df_playoff, on="Player", how="outer")

player_compare.sort_values("Player")

Finally, we can also add a column comparing our lists to the true list of Olympic Finalists for 2020.

In [ ]:
# Pull a dataset of Olympic finalists
olympics_finalists = df[df["OlympicsFinalist"] == 1][["Player", "OlympicsFinalist"]].drop_duplicates()

# Perform a left join with player_compare on "Player"
player_compare = player_compare.merge(olympics_finalists, on="Player", how="left")

# Sort the resulting DataFrame by Player (optional)
player_compare = player_compare.sort_values(by="Player")
player_compare

7 players (Anthony Davis, James Harden, Jimmy Butler, Kawhi Leonard, Kevin Durant, Lebron James, and Stephen Curry) are chosen in every variation of our model. Certain players only appear when we look at playoff statistics (e.g. Chris Paul, Draymond Green), while others look better when we focus on the regular season (e.g. Paul George, Damian Lillard).